In [1]:
# add more info (e.g., cross references) (Timing: ~ 2900 s)

# NOTE: this also adds predicted optimal temperature

# data stored in /supplData
# 'chem_xref.tsv' (https://www.metanetx.org/cgi-bin/mnxget/mnxref/chem_xref.tsv)
# 'reac_xref.tsv' (https://www.metanetx.org/cgi-bin/mnxget/mnxref/reac_xref.tsv)

import tarfile, re
import pandas as pd
import time
time_start = time.time()

keggdir = '/home/feiran' # the folder of the downloaded KEGG database

def getDictMNX(MNX2db, line, db):
    dbtmp = [re.findall(db+':\S+', line)[0].replace(db+':','')]
    MNXtmp = re.findall('MNX[RM]\d+', line)
    if len(MNXtmp) > 0:
        if MNXtmp[0] in MNX2db:
            MNX2db[MNXtmp[0]] += dbtmp # are there multiple MNX IDs to one KEGG ID?
        else:
            MNX2db[MNXtmp[0]] = dbtmp
    return MNX2db

def convertDict(kegg2MNX,MNX2db):
    kegg2db = dict()
    for key in kegg2MNX:
        dbvalue = []
        for MNX in kegg2MNX[key]:
            if MNX in MNX2db:
                dbvalue += MNX2db[MNX]
        kegg2db[key] = dbvalue
    return kegg2db


# Compound info
# SMILES and images
import os
import math
import pickle
import numpy as np
from rdkit import Chem
import tarfile
import glob
from rdkit.Chem import Draw
from pathlib import Path
# generate images from mol file and output smiles
exceptionlist = list()
compound_dict = dict()
tar = tarfile.open(keggdir + '/kegg/ligand/compound.tar.gz', 'r:gz')
mollist = [i for i in tar.getnames() if 'compound/mol/' in i]

def dump_file(dictionary, filename):
    with open(filename, 'wb') as file:
        pickle.dump(dictionary, file)
        
for molfile in mollist:
    try:
        stringWithMolData = tar.extractfile(molfile).read().decode()
        molstruct = Chem.MolFromMolBlock(stringWithMolData)
        #print(molstruct)
        p = Path(molfile)
        compound = p.stem
        #print(compound)
        compound_dict[compound] = Chem.MolToSmiles(molstruct)
        #print(compound_dict[compound])
        #Draw.MolToFile(molsturct, 'images/' + compound + '.png',size=(400,400))
        img=Draw.MolsToGridImage({molstruct}, molsPerRow=1, subImgSize=(300, 160), useSVG=True)
        with open('images/' + compound + '.svg', 'w') as f_handle:
            f_handle.write(img.data)
    except:
        p = Path(molfile)
        compound = p.stem
        exceptionlist.append(compound)

dump_file(compound_dict, 'output/kegg_smiles_dict.pickle')


# KEGG data
tar = tarfile.open(keggdir + '/kegg/ligand/compound.tar.gz', 'r:gz')
f = tar.extractfile('compound/compound')
cmpd_name = dict() # compound name (only the first name is used when there are multiple names)
cmpd_formula = dict() # compound formula
for line in f:
    if line.decode().startswith('ENTRY'):
        c_id = re.findall('[A-Z]\d{5}', line.decode())[0]
        cmpd_name[c_id] = []
        cmpd_formula[c_id] = []
    elif line.decode().startswith('NAME'):
        cmpd_name[c_id] = [line.decode().replace('NAME','').strip().replace(';','')]
    elif line.decode().startswith('FORMULA'):
        cmpd_formula[c_id] = [line.decode().replace('FORMULA','').strip().replace(';','')]
f.close()
tar.close()
# Other sources
f = open('supplData/chem_xref.tsv','r')
keggC2MNXM = dict()
MNXM2seedM = dict()
MNXM2biggM = dict()
MNXM2chebi = dict()
MNXM2metacycM = dict()
MNXM2sabiorkM = dict()
MNXM2reactomeM = dict()
for line in f:
    if line.startswith('kegg') and line.find('secondary/obsolete/fantasy identifier') == -1:
        keggC = re.findall('[A-Z]\d{5}', line)[0]
        MNXM = re.findall('MNXM\d+', line)
        if keggC in keggC2MNXM:
            keggC2MNXM[keggC] += MNXM # are there multiple MNXM IDs to one KEGG ID?
            keggC2MNXM[keggC] = list(set(keggC2MNXM[keggC]))
        else:
            keggC2MNXM[keggC] = MNXM
    elif line.startswith('seedM') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXM2seedM = getDictMNX(MNXM2seedM, line, 'seedM')
    elif line.startswith('biggM') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXM2biggM = getDictMNX(MNXM2biggM, line, 'biggM')
    elif line.startswith('chebi') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXM2chebi = getDictMNX(MNXM2chebi, line, 'chebi')
    elif line.startswith('metacycM') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXM2metacycM = getDictMNX(MNXM2metacycM, line, 'metacycM')
    elif line.startswith('sabiorkM') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXM2sabiorkM = getDictMNX(MNXM2sabiorkM, line, 'sabiorkM')
    elif line.startswith('reactomeM') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXM2reactomeM = getDictMNX(MNXM2reactomeM, line, 'reactomeM')
keggC2seedM = convertDict(keggC2MNXM,MNXM2seedM)
keggC2biggM = convertDict(keggC2MNXM,MNXM2biggM)
keggC2chebi = convertDict(keggC2MNXM,MNXM2chebi)
keggC2metacycM = convertDict(keggC2MNXM,MNXM2metacycM)
keggC2sabiorkM = convertDict(keggC2MNXM,MNXM2sabiorkM)
keggC2reactomeM = convertDict(keggC2MNXM,MNXM2reactomeM)
# write file
fout = open('supplOutput/compound.txt', 'w')
fout.write('KEGG\t'+'Name\t'+'Formula\t'+'MetaNetX\t'+'ModelSEED\t'+'BiGG\t'+'ChEBI\t'+'MetaCyc\t'+'SABIO-RK\t'+'Reactome\t'+'SMILES'+'\n')
for key in cmpd_name:
    if key in compound_dict:
        smile = compound_dict[key]
    else:
        smile = ''
    if key in keggC2MNXM:
        line_to_write = key + '\t' + ';'.join(cmpd_name[key]) + '\t' + ';'.join(cmpd_formula[key]) + '\t' + ';'.join(keggC2MNXM[key]) + '\t'\
        + ';'.join(keggC2seedM[key]) + '\t' + ';'.join(keggC2biggM[key]) + '\t' + ';'.join(keggC2chebi[key]) + '\t'\
        + ';'.join(keggC2metacycM[key]) + '\t' + ';'.join(keggC2sabiorkM[key]) + '\t' + ';'.join(keggC2reactomeM[key]) + '\t' + smile + '\n'
    else:
        line_to_write = key + '\t' + ';'.join(cmpd_name[key]) + '\t' + ';'.join(cmpd_formula[key]) + '\t\t\t\t\t\t\t' + '\t' + smile + '\n'
    fout.write(line_to_write)
fout.close()


# Reaction info
# KEGG data
tar = tarfile.open(keggdir + '/kegg/ligand/reaction.tar.gz', 'r:gz')
f = tar.extractfile('reaction/reaction')
rxn_name = dict() # reaction name (only the first name is used when there are multiple names)
rxn_formula = dict() # reaction formula
for line in f:
    if line.decode().startswith('ENTRY'):
        r_id = re.findall('R\d{5}', line.decode())[0]
        rxn_name[r_id] = []
        rxn_formula[r_id] = []
    elif line.decode().startswith('NAME'):
        rxn_name[r_id] = [line.decode().replace('NAME','').strip().replace(';','')]
    elif line.decode().startswith('DEFINITION'):
        rxn_formula[r_id] = [line.decode().replace('DEFINITION','').strip().replace(';','')]
f.close()
tar.close()
# Other sources
f = open('supplData/reac_xref.tsv','r')
keggR2MNXR = dict()
MNXR2rheaR = dict()
MNXR2seedR = dict()
MNXR2biggR = dict()
MNXR2metacycR = dict()
MNXR2sabiorkR = dict()
for line in f:
    if line.startswith('keggR'):
        keggC = re.findall('keggR:R\d{5}', line)[0].replace('keggR:','')
        MNXM = re.findall('MNXR\d+', line)
        if keggC in keggR2MNXR: 
            keggR2MNXR[keggC] += MNXM # are there multiple MNXR IDs to one KEGG ID?
        else:
            keggR2MNXR[keggC] = MNXM
    elif line.startswith('rheaR') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXR2rheaR = getDictMNX(MNXR2rheaR, line, 'rheaR')
    elif line.startswith('seedR') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXR2seedR = getDictMNX(MNXR2seedR, line, 'seedR')
    elif line.startswith('biggR') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXR2biggR = getDictMNX(MNXR2biggR, line, 'biggR')
    elif line.startswith('metacycR') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXR2metacycR = getDictMNX(MNXR2metacycR, line, 'metacycR')
    elif line.startswith('sabiorkR') and line.find('secondary/obsolete/fantasy identifier') == -1:
        MNXR2sabiorkR = getDictMNX(MNXR2sabiorkR, line, 'sabiorkR')
keggR2rheaR = convertDict(keggR2MNXR,MNXR2rheaR)
keggR2seedR = convertDict(keggR2MNXR,MNXR2seedR)
keggR2biggR = convertDict(keggR2MNXR,MNXR2biggR)
keggR2metacycR = convertDict(keggR2MNXR,MNXR2metacycR)
keggR2sabiorkR = convertDict(keggR2MNXR,MNXR2sabiorkR)
# write file
fout = open('supplOutput/reaction.txt', 'w')
fout.write('KEGG\t'+'Name\t'+'Equation\t'+'MetaNetX\t'+'Rhea\t'+'ModelSEED\t'+'BiGG\t'+'MetaCyc\t'+'SABIO-RK'+'\n')
for key in rxn_name:
    if key in keggR2MNXR:
        line_to_write = key + '\t' + ';'.join(rxn_name[key]) + '\t' + ';'.join(rxn_formula[key]) + '\t' + ';'.join(keggR2MNXR[key]) + '\t'\
        + ';'.join(keggR2rheaR[key]) + '\t' + ';'.join(keggR2seedR[key]) + '\t' + ';'.join(keggR2biggR[key]) + '\t'\
        + ';'.join(keggR2metacycR[key]) + '\t' + ';'.join(keggR2sabiorkR[key]) + '\n'
    else:
        line_to_write = key + '\t' + ';'.join(rxn_name[key]) + '\t' + ';'.join(rxn_formula[key]) + '\t\t\t\t\t\t\n'
    fout.write(line_to_write)
fout.close()


# Enzyme info
fout = open('supplOutput/ec.txt', 'w')
fout.write('EC' + '\t' + 'Name' + '\n')
# KEGG data
tar = tarfile.open(keggdir + '/kegg/ligand/enzyme.tar.gz', 'r:gz')
f = tar.extractfile('enzyme/enzyme')
ec_name = dict() # EC number name
flag = False # used for names in multiple lines
for line in f:
    if line.decode().startswith('ENTRY'):
        ec = re.findall('\S+\.\S+\.\S+\.\S+', line.decode())[0]
        ec_name[ec] = []
    elif line.decode().startswith('NAME'):
        ec_name[ec] = line.decode().replace('NAME','').strip().replace(';','')
        if line.decode().endswith(';\n'):
            flag = True
    # for names in multiple lines
    elif flag and line.decode().startswith(' '):
        ec_name[ec] += ';' + line.decode().strip().replace(';','')
        if line.decode().endswith(';\n'):
            flag = True
        else:
            flag = False
    elif line.decode().startswith('///'):
        line_to_write = ec + '\t' + ec_name[ec] + '\n'
        fout.write(line_to_write)
f.close()
tar.close()
fout.close()


# Organism info
org2tax = dict()
fhand = open(keggdir + '/kegg/genes/misc/taxonomic_rank')
for line in fhand:
    if not line.startswith('#'):
        org2tax[line.split('\t')[0]] = line.split('\t')[1]
fhand.close()

fhand = open(keggdir + '/kegg/genes/misc/taxonomy')
fout = open('supplOutput/organism.txt', 'w')
fout.write('Org code' + '\t' + 'Entry' + '\t' + 'Name' + '\t' + 'NCBI Taxonomy' + '\n')
for line in fhand:
    if not line.startswith('#'):
        if line.split('\t')[1] in org2tax:
            tax = org2tax[line.split('\t')[1]]
        else:
            tax = ''
        line_to_write = line.split('\t')[1] + '\t' + line.split('\t')[2] + '\t' + line.split('\t')[3].rstrip() + '\t' + tax + '\n'
        fout.write(line_to_write)
fhand.close()
fout.close()


# Domain info
fout = open('supplOutput/domain.txt', 'w')
fout.write('Abbreviation' + '\t' + 'Full name' + '\n')
line_to_write = 'A\tArchaea\n' + 'B\tBacteria\n' + 'E\tEukaryotes\n'
fout.write(line_to_write)
fout.close()


<>:17: SyntaxWarning: invalid escape sequence '\S'
<>:18: SyntaxWarning: invalid escape sequence '\d'
<>:87: SyntaxWarning: invalid escape sequence '\d'
<>:107: SyntaxWarning: invalid escape sequence '\d'
<>:108: SyntaxWarning: invalid escape sequence '\d'
<>:158: SyntaxWarning: invalid escape sequence '\d'
<>:177: SyntaxWarning: invalid escape sequence '\d'
<>:178: SyntaxWarning: invalid escape sequence '\d'
<>:222: SyntaxWarning: invalid escape sequence '\S'
<>:17: SyntaxWarning: invalid escape sequence '\S'
<>:18: SyntaxWarning: invalid escape sequence '\d'
<>:87: SyntaxWarning: invalid escape sequence '\d'
<>:107: SyntaxWarning: invalid escape sequence '\d'
<>:108: SyntaxWarning: invalid escape sequence '\d'
<>:158: SyntaxWarning: invalid escape sequence '\d'
<>:177: SyntaxWarning: invalid escape sequence '\d'
<>:178: SyntaxWarning: invalid escape sequence '\d'
<>:222: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_2125143/3909446662.py:17: SyntaxWarning: invalid escape

In [4]:
import os
import time
from Bio import SeqIO
import glob
import gzip
import tarfile
import pandas as pd

# --- 配置 ---
keggdir = '/home/feiran' # KEGG 数据库的根目录
output_dir = 'supplOutput/gene' # 输出目录

# --- 函数定义 ---
def getProtCrossRef(org, base_dir, out_dir):
    """
    为给定的生物体(org)提取KEGG基因ID，并交叉引用NCBI和UniProt ID。
    将结果写入到指定的输出目录。
    """
    org_path = os.path.join(base_dir, 'kegg/genes/organisms', org)

    ### 1. 获取蛋白质列表
    fasta_files = glob.glob(os.path.join(org_path, "*.pep.gz"))
    if not fasta_files:
        print(f"警告: 在 {org_path} 中未找到 .pep.gz 文件，跳过生物体 {org}。")
        return

    fastafile = fasta_files[0]
    protList = []
    
    # --- 这里是关键修改 ---
    # 使用 try...except 块来捕获 BadGzipFile 等文件读取错误
    try:
        with gzip.open(fastafile, "rt") as handle:
            for record in SeqIO.parse(handle, "fasta"):
                protList.append(record.id.replace(org + ':', ''))
    except gzip.BadGzipFile:
        # 当文件不是有效的gzip文件时，捕获错误
        print(f"错误: 文件 {fastafile} 是一个无效或损坏的 Gzip 文件。跳过生物体 {org}。")
        return # 直接返回，不再处理这个生物体
    except Exception as e:
        # 捕获其他可能的读取错误
        print(f"错误: 读取fasta文件 {fastafile} 时出现未知错误: {e}。跳过生物体 {org}。")
        return

    # 如果文件是空的，protList会是空的，这没有问题，后续会生成一个空的结果文件
    # if not protList:
    #     print(f"信息: 文件 {fastafile} 中没有找到蛋白质序列。")
    
    ### 2. 获取NCBI和UniProt的交叉引用信息 (这部分代码保持不变)
    link_tar_path = os.path.join(org_path, f'{org}_link.tar.gz')
    prot2ncbi = {}
    prot2up = {}

    if not os.path.exists(link_tar_path):
        print(f"警告: 找不到链接文件 {link_tar_path}，跳过交叉引用。")
    else:
        # --- 关键修改：自动判断是否为 gzip 压缩 ---
        try:
            # 先尝试以 'r:gz' 模式打开（gzip 压缩）
            with tarfile.open(link_tar_path, 'r:gz') as tar:
                pass  # 能打开说明是 gzip 文件
            mode = 'r:gz'
        except tarfile.ReadError:
            try:
                # 再尝试以 'r:' 模式打开（未压缩 tar）
                with tarfile.open(link_tar_path, 'r:') as tar:
                    pass
                mode = 'r:'
            except tarfile.ReadError:
                print(f"错误: 无法读取tar文件 {link_tar_path}: 不是有效的 tar 或 tar.gz 文件")
                return

        # 使用正确模式打开
        with tarfile.open(link_tar_path, mode) as tar:
            try:
                f_ncbiPid = tar.extractfile(f'{org}_ncbi-proteinid.list')
                if f_ncbiPid:
                    with f_ncbiPid:
                        info = pd.read_table(f_ncbiPid, sep='\t', header=None)
                        df_ncbi = pd.concat([info[0].str.replace(org + ':', ''), info[1].str.replace('ncbi-proteinid:', '')], axis=1)
                        df_ncbi.columns = ['KEGG_gene_id', 'NCBI_protein_id']
                        prot2ncbi = df_ncbi.groupby('KEGG_gene_id')['NCBI_protein_id'].apply(list).to_dict()
            except KeyError:
                pass  # 文件不存在

            try:
                f_uniprotPid = tar.extractfile(f'{org}_uniprot.list')
                if f_uniprotPid:
                    with f_uniprotPid:
                        info = pd.read_table(f_uniprotPid, sep='\t', header=None)
                        df_uniprot = pd.concat([info[0].str.replace(org + ':', ''), info[1].str.replace('up:', '')], axis=1)
                        df_uniprot.columns = ['KEGG_gene_id', 'Uniprot_id']
                        prot2up = df_uniprot.groupby('KEGG_gene_id')['Uniprot_id'].apply(list).to_dict()
            except KeyError:
                pass

    
    ### 3. 写入文件 (这部分代码保持不变)
    output_file = os.path.join(out_dir, f'{org}.txt')
    try:
        with open(output_file, 'w') as fout:
            for key in protList:
                ncbi = ';'.join(prot2ncbi.get(key, []))
                up = ';'.join(prot2up.get(key, []))
                line_to_write = f"{key}\t{ncbi}\t{up}\n"
                fout.write(line_to_write)
    except IOError as e:
        print(f"错误: 无法写入文件 {output_file}: {e}")


# --- 主程序 (保持不变) ---
def main():
    # 你的主程序代码...
    print("脚本开始执行...")
    time_start = time.time()
    os.makedirs(output_dir, exist_ok=True)
    
    header_file = os.path.join(output_dir, 'gene.header')
    with open(header_file, 'w') as fout:
        fout.write('KEGG gene id\tNCBI protein id\tUniProt id\n')
    
    organisms_path = os.path.join(keggdir, 'kegg/genes/organisms/')
    try:
        org_dirs = [d.name for d in os.scandir(organisms_path) if d.is_dir()]
        print(f"找到 {len(org_dirs)} 个生物体进行处理。")
        for org in org_dirs:
            # print(f"正在处理: {org}...")
            getProtCrossRef(org, keggdir, output_dir)
    except FileNotFoundError:
        print(f"错误: 找不到生物体目录 {organisms_path}。请检查 `keggdir` 路径是否正确。")
        return

    time_end = time.time()
    print(f"\n脚本执行完毕。")
    print(f"总耗时: {round(time_end - time_start)} 秒")

# # 脚本入口
# if __name__ == "__main__":
#     main()

In [5]:
def retry():
    # 你的主程序代码...
    print("脚本开始执行...")
    time_start = time.time()
    os.makedirs(output_dir, exist_ok=True)
    
    header_file = os.path.join(output_dir, 'gene.header')
    with open(header_file, 'w') as fout:
        fout.write('KEGG gene id\tNCBI protein id\tUniProt id\n')
    
    organisms_path = os.path.join(keggdir, 'kegg/genes/organisms/')
    try:
        org_dirs = [d.name for d in os.scandir(organisms_path) if d.is_dir()]
        print(f"找到 {len(org_dirs)} 个生物体进行处理。")
        org_dirs = ['rge', 'bpk', 'fcy', 'pdul', 'mgy', 'lch', 'nul', 'eae'] # 只处理这些物种
        for org in org_dirs:
            # print(f"正在处理: {org}...")
            getProtCrossRef(org, keggdir, output_dir)
    except FileNotFoundError:
        print(f"错误: 找不到生物体目录 {organisms_path}。请检查 `keggdir` 路径是否正确。")
        return

    time_end = time.time()
    print(f"\n脚本执行完毕。")
    print(f"总耗时: {round(time_end - time_start)} 秒")
retry()

脚本开始执行...
找到 10915 个生物体进行处理。

脚本执行完毕。
总耗时: 3 秒
